<h1 align="center"> Lingüística Computacional - MIARFID</h1>
<h1 align="center"> Trabajo final - Transformers</h1>

**Estudiantes:** Carlos Broch Cubedo · Ferran Artero Merino

# 0. Configuración

In [1]:
!pip -q install iterative-stratification peft
!pip -q uninstall -y torchao

In [2]:
import os, re, ast, gc, json, math, time, zipfile, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold
from google.colab import drive

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/LC_TrabajoFinal"
OUT_DIR = os.path.join(DATA_DIR, "outputs", "transformers")
RUNS_DIR = os.path.join(OUT_DIR, "runs")       # un subdirectorio por experimento con foldK.npz
EMB_DIR = os.path.join(OUT_DIR, "embeddings")
for d in (OUT_DIR, RUNS_DIR, EMB_DIR):
    os.makedirs(d, exist_ok=True)
TRAIN_FILE = os.path.join(DATA_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(DATA_DIR, "dataset_test.csv")

Mounted at /content/drive


In [3]:
HOLDOUT_SIZE, N_FOLDS = 0.15, 3
SMART_MINORITY_MAX = 1000
MAX_LABELS = 4

DEFAULTS = dict(
    model="roberta-base",
    max_len=384,
    epochs=4,
    batch_size=16,
    grad_accum=2,             # lote efectivo 32
    lr=3e-5,                  # encoder
    head_lr=3e-4,             # cabeza: 10x
    weight_decay=0.01,
    warmup=0.06,
    loss="bce",               # "bce" | "bce_pw" (pos_weight) | "asl"
    method="full",            # "full" | "lora"
    llrd=None,                # decaimiento de la LR por capa (p. ej. 0.9)
    lora_r=16,
    pooling="cls",            # "cls" | "last" (Qwen3)
    dropout=0.1,
    train_dedup="full",
    grad_ckpt=False,
)

# 1. Datos y partición
Mismo preprocesado, holdout y particiones que los modelos clásicos. Para evaluar con una sola partición, su validación se divide en dos mitades para ajustar los umbrales por *cross-fitting*.

In [4]:
CLASSES = ['cs.AI', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DC', 'cs.GR', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG',
           'cs.MA', 'cs.MM', 'cs.NE', 'cs.RO', 'cs.SI', 'cs.SY', 'eess.IV', 'eess.SP', 'eess.SY', 'math.IT',
           'math.OC', 'math.ST', 'physics.chem-ph', 'stat.AP', 'stat.ME', 'stat.ML', 'stat.TH']
L = len(CLASSES)

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\$[^$]*\$", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def clean(s):
    return re.sub(r"\s+", " ", str(s)).strip()

raw = pd.read_csv(TRAIN_FILE)
raw["labels"] = raw["terms"].map(ast.literal_eval)
raw["title_n"], raw["summary_n"] = raw["titles"].map(norm), raw["summaries"].map(norm)
data = (raw.groupby(["title_n", "summary_n"], sort=False)
           .agg(titles=("titles", "first"), summaries=("summaries", "first"),
                labels=("labels", lambda ls: sorted({l for x in ls for l in x})),
                n_copies=("titles", "size"))
           .reset_index(drop=True))
data["text"] = data["titles"].map(clean) + ". " + data["summaries"].map(clean)

mlb = MultiLabelBinarizer(classes=CLASSES)
Y = mlb.fit_transform(data["labels"])
N = len(data)
N_COPIES = data["n_copies"].to_numpy()

test = pd.read_csv(TEST_FILE)
TEST_TEXTS = (test["titles"].map(clean) + ". " + test["summaries"].map(clean)).tolist()

ROOTS = ["cs.CV", "cs.LG", "stat.ML"]
ROOT_IDX = np.array([CLASSES.index(r) for r in ROOTS])
ALIAS_PAIRS = [(a, b) for i, a in enumerate(CLASSES) for b in CLASSES[i + 1:]
               if (Y[:, CLASSES.index(a)] == Y[:, CLASSES.index(b)]).all()]
freq = pd.Series(Y.sum(0), index=CLASSES)
MINORITY_IDX = np.array([CLASSES.index(c) for c in freq.index[freq < SMART_MINORITY_MAX]])

def copies_to_keep(mode):
    if mode == "none":
        return N_COPIES.copy()
    if mode == "full":
        return np.ones(N, dtype=int)
    return np.where(Y[:, MINORITY_IDX].any(1), N_COPIES, 1)

dev_idx, ho_idx = next(MultilabelStratifiedShuffleSplit(n_splits=1, test_size=HOLDOUT_SIZE, random_state=SEED)
                       .split(np.zeros(N), Y))
Y_dev, Y_ho = Y[dev_idx], Y[ho_idx]
FOLDS = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
             .split(np.zeros(len(dev_idx)), Y_dev))
FOLD_ID = np.empty(len(dev_idx), dtype=int)
HALF_ID = np.full(len(dev_idx), -1)
for k, (_, va) in enumerate(FOLDS):
    FOLD_ID[va] = k
    halves = MultilabelStratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
    for h, (_, part) in enumerate(halves.split(np.zeros(len(va)), Y_dev[va])):
        HALF_ID[va[part]] = 2 * k + h

print(f"Artículos únicos: {N} | dev: {len(dev_idx)} | holdout: {len(ho_idx)} | test: {len(TEST_TEXTS)}")

Artículos únicos: 32577 | dev: 27674 | holdout: 4903 | test: 4885


# 2. Funciones
### Umbrales, posproceso y evaluación
Mismo umbral por clase con *cross-fitting* y mismo posproceso que los clásicos. Cada experimento guarda por partición sus puntuaciones de validación, holdout y test.

In [5]:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "precision": precision,
            "recall": recall, "hamming_loss": hamming_loss(labels, preds)}

def tie_alias(S):
    S = np.array(S, dtype=np.float32, copy=True)
    for a, b in ALIAS_PAIRS:
        i, j = CLASSES.index(a), CLASSES.index(b)
        S[:, i] = S[:, j] = (S[:, i] + S[:, j]) / 2
    return S

def tune_thresholds(Yt, S, n_grid=200):
    thr = np.zeros(S.shape[1])
    for j in range(S.shape[1]):
        y, s = Yt[:, j].astype(bool), S[:, j]
        cand = np.unique(np.r_[np.quantile(s, np.linspace(0, 1, n_grid)), 0.0])
        pred = s[None, :] >= cand[:, None]
        tp = (pred & y).sum(1)
        fp = (pred & ~y).sum(1)
        fn = y.sum() - tp
        thr[j] = cand[(2 * tp / np.maximum(2 * tp + fp + fn, 1)).argmax()]
    return thr

def crossfit_thresholds(Yt, S, group_id):
    T = np.zeros_like(S)
    for g in np.unique(group_id):
        m = group_id == g
        T[m] = tune_thresholds(Yt[~m], S[~m])
    return T

def decode_flat(S, T):
    M = S - T
    P = (M >= 0).astype(int)
    no_root = P[:, ROOT_IDX].sum(1) == 0
    best = ROOT_IDX[M[:, ROOT_IDX].argmax(1)]
    P[no_root, best[no_root]] = 1
    for i in np.where(P.sum(1) > MAX_LABELS)[0]:
        on = np.where(P[i])[0]
        P[i, on[np.argsort(-M[i, on])[MAX_LABELS:]]] = 0
    for a, b in ALIAS_PAIRS:
        i, j = CLASSES.index(a), CLASSES.index(b)
        P[:, i] = P[:, j] = np.maximum(P[:, i], P[:, j])
    return P

In [6]:
def run_dir(name):
    return os.path.join(RUNS_DIR, name)

def fold_path(name, k):
    return os.path.join(run_dir(name), f"fold{k}.npz")

def fold_done(name, k):
    return os.path.exists(fold_path(name, k))

def save_config(name, cfg):
    os.makedirs(run_dir(name), exist_ok=True)
    path = os.path.join(run_dir(name), "config.json")
    if not os.path.exists(path):
        with open(path, "w") as fh:
            json.dump(cfg, fh, indent=2, default=str)

def save_fold(name, k, S_va, S_ho, S_te, secs):
    np.savez(fold_path(name, k), S_va=S_va.astype(np.float32), S_ho=S_ho.astype(np.float32),
             S_te=S_te.astype(np.float32), secs=secs)

def load_run(name):
    folds = [k for k in range(N_FOLDS) if fold_done(name, k)]
    if not folds:
        raise FileNotFoundError(name)
    oof = np.full((len(dev_idx), L), np.nan, dtype=np.float32)
    ho, te, secs = [], [], 0.0
    for k in folds:
        z = np.load(fold_path(name, k))
        oof[FOLDS[k][1]] = z["S_va"]
        ho.append(z["S_ho"]); te.append(z["S_te"]); secs += float(z["secs"])
    cfg_path = os.path.join(run_dir(name), "config.json")
    cfg = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}
    return {"name": name, "folds": folds, "oof": oof, "ho": np.mean(ho, 0), "te": np.mean(te, 0),
            "secs": secs, "cfg": cfg}

def evaluate_run(R, folds=None):
    # Una partición: cross-fitting entre sus dos mitades. Varias: entre particiones
    folds = R["folds"] if folds is None else folds
    rows = np.isin(FOLD_ID, folds)
    S, Yt = R["oof"][rows], Y_dev[rows]
    gid = (FOLD_ID if len(folds) > 1 else HALF_ID)[rows]
    P = decode_flat(S, crossfit_thresholds(Yt, S, gid))
    out = compute_metrics(Yt, P)
    out["f1_std"] = float(np.std([f1_score(Yt[gid == g], P[gid == g], average="macro", zero_division=0)
                                  for g in np.unique(gid)]))
    out["per_class_f1"] = f1_score(Yt, P, average=None, zero_division=0)
    return out

def holdout_eval(R):
    rows = np.isin(FOLD_ID, R["folds"])
    return compute_metrics(Y_ho, decode_flat(R["ho"], tune_thresholds(Y_dev[rows], R["oof"][rows])))

def results_table(names):
    rows = []
    for n in names:
        try:
            R = load_run(n)
        except FileNotFoundError:
            continue
        row = {"experimento": n, "modelo": R["cfg"].get("model", R["cfg"].get("tipo", "")), "folds": len(R["folds"])}
        if 0 in R["folds"]:
            m0 = evaluate_run(R, [0])
            row["F1 fold0"], row["±"] = m0["f1"], m0["f1_std"]
        if len(R["folds"]) == N_FOLDS:
            m = evaluate_run(R)
            row.update({"F1 CV": m["f1"], "± CV": m["f1_std"], "precision": m["precision"], "recall": m["recall"],
                        "accuracy": m["accuracy"], "hamming": m["hamming_loss"]})
        row["F1 holdout"] = holdout_eval(R)["f1"]
        row["horas"] = R["secs"] / 3600
        rows.append(row)
    return pd.DataFrame(rows)

### Modelo y entrenamiento
Encoder preentrenado, vector del primer token (último en Qwen3), dropout y una capa lineal de 28 salidas con el sesgo inicializado a la frecuencia de cada clase. Se conserva el modelo de la última época.

In [7]:
class Classifier(nn.Module):
    def __init__(self, name, n_out, pooling="cls", dropout=0.1):
        super().__init__()
        config = AutoConfig.from_pretrained(name)
        if hasattr(config, "reference_compile"):
            config.reference_compile = False
        self.encoder = AutoModel.from_pretrained(name, config=config).float()
        self.n_layers = self.encoder.config.num_hidden_layers
        self.pooling = pooling
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, n_out)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "last":
            z = h[torch.arange(h.size(0), device=h.device), attention_mask.sum(1) - 1]
        else:
            z = h[:, 0]
        return self.head(self.drop(z.float())), z

_TOK_CACHE = {}
def get_tokens(model_name, max_len, add_eos=False):
    key = (model_name, max_len, add_eos)
    if key not in _TOK_CACHE:
        _TOK_CACHE.clear()
        tok = AutoTokenizer.from_pretrained(model_name)
        eos = tok.eos_token_id
        def fix(x):
            if add_eos and eos is not None and (not x or x[-1] != eos):
                x = x[:max_len - 1] + [eos]
            return np.array(x, dtype=np.int32)
        enc = lambda texts: [fix(x) for x in tok(texts, truncation=True, max_length=max_len)["input_ids"]]
        _TOK_CACHE[key] = (tok, enc(data["text"].tolist()), enc(TEST_TEXTS))
    return _TOK_CACHE[key]

def pad_id_of(tok):
    return tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def make_batches(lengths, bs, shuffle, rng=None):
    # Lotes de longitud parecida para reducir el padding
    idx = np.arange(len(lengths))
    if not shuffle:
        idx = idx[np.argsort(lengths, kind="stable")]
        return [idx[i:i + bs] for i in range(0, len(idx), bs)]
    idx = rng.permutation(idx)
    batches = []
    for c in range(0, len(idx), bs * 50):
        chunk = idx[c:c + bs * 50]
        chunk = chunk[np.argsort(lengths[chunk], kind="stable")]
        batches += [chunk[i:i + bs] for i in range(0, len(chunk), bs)]
    return [batches[i] for i in rng.permutation(len(batches))]

def collate(seqs, pad_id, side="right"):
    m = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), m), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), m), dtype=torch.long)
    for i, s in enumerate(seqs):
        t = torch.from_numpy(s.astype(np.int64))
        if side == "right":
            ids[i, :len(s)], mask[i, :len(s)] = t, 1
        else:
            ids[i, m - len(s):], mask[i, m - len(s):] = t, 1
    return ids.to(DEVICE), mask.to(DEVICE)

@torch.no_grad()
def predict(model, seqs, pad_id, bs=64):
    model.eval()
    S = np.zeros((len(seqs), model.head.out_features), np.float32)
    Z = np.zeros((len(seqs), model.head.in_features), np.float16)
    lens = np.array([len(s) for s in seqs])
    for b in make_batches(lens, bs, shuffle=False):
        ids, mask = collate([seqs[i] for i in b], pad_id)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits, z = model(ids, mask)
        S[b] = logits.float().cpu().numpy()
        Z[b] = z.float().cpu().numpy()
    return S, Z

In [8]:
def asl_loss(logits, y, gamma_neg=4.0, clip=0.05, eps=1e-8):
    p = torch.sigmoid(logits)
    p_neg = (p - clip).clamp(min=0)
    loss_pos = y * torch.log(p.clamp(min=eps))
    loss_neg = (1 - y) * p_neg ** gamma_neg * torch.log((1 - p_neg).clamp(min=eps))
    return -(loss_pos + loss_neg).mean()

def make_loss(kind, prior):
    if kind == "bce":
        return lambda z, y: F.binary_cross_entropy_with_logits(z, y)
    if kind == "bce_pw":
        # pos_weight = sqrt(n_neg / n_pos), recortado a [1, 30]
        pw = torch.tensor(np.clip(np.sqrt((1 - prior) / prior), 1.0, 30.0), dtype=torch.float32, device=DEVICE)
        return lambda z, y: F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)
    return asl_loss

def layer_id(name, n_layers):
    m = re.search(r"\.(?:layer|layers)\.(\d+)\.", name)
    if m:
        return int(m.group(1)) + 1
    return 0 if "embed" in name else n_layers + 1

def param_groups(model, cfg):
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        no_decay = n.endswith("bias") or "norm" in n.lower()
        if n.startswith("head."):
            lr = cfg["head_lr"]
        else:
            lr = cfg["lr"] * (cfg["llrd"] ** (model.n_layers + 1 - layer_id(n, model.n_layers)) if cfg["llrd"] else 1.0)
        groups.setdefault((lr, no_decay), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": 0.0 if nd else cfg["weight_decay"]}
            for (lr, nd), ps in groups.items()]

def add_lora(encoder, r):
    from peft import LoraConfig, get_peft_model
    return get_peft_model(encoder, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.1, target_modules="all-linear"))

In [9]:
def train_fold(name, cfg, k):
    t0 = time.time()
    torch.manual_seed(SEED + k)
    rng = np.random.default_rng(SEED + k)
    tok, ids_all, ids_test = get_tokens(cfg["model"], cfg["max_len"], add_eos=cfg["pooling"] == "last")
    pad = pad_id_of(tok)

    tr, va = FOLDS[k]
    tr_rows, va_rows = dev_idx[tr], dev_idx[va]
    rep = np.repeat(tr_rows, copies_to_keep(cfg["train_dedup"])[tr_rows])
    Y_tr = torch.tensor(Y[rep], dtype=torch.float32)
    seqs_tr = [ids_all[i] for i in rep]
    lens_tr = np.array([len(s) for s in seqs_tr])

    model = Classifier(cfg["model"], L, cfg["pooling"], cfg["dropout"]).to(DEVICE)
    prior = Y_tr.numpy().mean(0).clip(1e-4, 1 - 1e-4)
    with torch.no_grad():
        model.head.bias.copy_(torch.tensor(np.log(prior / (1 - prior)), dtype=torch.float32))
    if cfg["grad_ckpt"]:
        model.encoder.config.use_cache = False
        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if cfg["method"] == "lora":
        model.encoder = add_lora(model.encoder, cfg["lora_r"])
    for n, p in model.encoder.named_parameters():
        p.requires_grad = cfg["method"] != "lora" or "lora_" in n
    loss_fn = make_loss(cfg["loss"], prior)

    opt = torch.optim.AdamW(param_groups(model, cfg))
    n_steps = cfg["epochs"] * math.ceil(math.ceil(len(rep) / cfg["batch_size"]) / cfg["grad_accum"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * n_steps), n_steps)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    trainable = [p for p in model.parameters() if p.requires_grad]

    for ep in range(cfg["epochs"]):
        model.train()
        total, batches = 0.0, make_batches(lens_tr, cfg["batch_size"], True, rng)
        opt.zero_grad(set_to_none=True)
        for step, b in enumerate(batches):
            ids, mask = collate([seqs_tr[i] for i in b], pad)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
                logits, _ = model(ids, mask)
            loss = loss_fn(logits.float(), Y_tr[b].to(DEVICE))
            scaler.scale(loss / cfg["grad_accum"]).backward()
            if (step + 1) % cfg["grad_accum"] == 0 or step == len(batches) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
            total += loss.item()
        print(f"   época {ep + 1}: loss {total / len(batches):.5f} ({(time.time() - t0) / 60:.1f} min)")

    S_va, Z_va = predict(model, [ids_all[i] for i in va_rows], pad)
    S_ho, Z_ho = predict(model, [ids_all[i] for i in ho_idx], pad)
    S_te, Z_te = predict(model, ids_test, pad)
    _, Z_tr = predict(model, [ids_all[i] for i in tr_rows], pad)
    np.savez(os.path.join(run_dir(name), f"feat_fold{k}.npz"), tr=Z_tr, va=Z_va, ho=Z_ho, te=Z_te)
    save_fold(name, k, tie_alias(S_va), tie_alias(S_ho), tie_alias(S_te), time.time() - t0)
    del model, opt
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def run_experiment(name, folds=(0,)):
    # Las particiones ya guardadas en Drive no se vuelven a entrenar
    cfg = {**DEFAULTS, **EXPERIMENTS[name]}
    save_config(name, cfg)
    for k in folds:
        if not fold_done(name, k):
            print(f"[{name}] partición {k}: entrenando {cfg['model']}")
            train_fold(name, cfg, k)

# 3. Catálogo de experimentos
Seis modelos ajustados con la misma receta y, sobre los tres mejores (SPECTER2, Qwen3 + LoRA y SciBERT), las ablaciones de la memoria: `pos_weight`, LoRA y LLRD.

In [10]:
EXPERIMENTS = {
    "E1_roberta":    {"model": "roberta-base"},
    "E2_scibert":    {"model": "allenai/scibert_scivocab_uncased"},
    "E3_specter2":   {"model": "allenai/specter2_base"},
    "E4_modernbert": {"model": "answerdotai/ModernBERT-base", "lr": 5e-5},
    "E5_deberta":    {"model": "microsoft/deberta-v3-base", "lr": 2e-5},
    "E6_qwen3_lora": {"model": "Qwen/Qwen3-Embedding-0.6B", "method": "lora", "pooling": "last", "lr": 2e-4,
                      "head_lr": 1e-3, "epochs": 3, "batch_size": 8, "grad_accum": 4, "grad_ckpt": True},
}
SCREEN = list(EXPERIMENTS)

# Mejor modelo del ajuste fino -> (prefijo, LLRD). Qwen3 ya se ajusta con LoRA
TOP_MODELS = {"E3_specter2": ("", 0.9), "E6_qwen3_lora": ("QW_", 0.95), "E2_scibert": ("SB_", 0.9)}
TOP_NAMES = {"E3_specter2": "SPECTER2", "E6_qwen3_lora": "Qwen3 + LoRA", "E2_scibert": "SciBERT"}
ABLATIONS = {}
for base, (prefix, llrd) in TOP_MODELS.items():
    abl = {"A1_posweight": dict(loss="bce_pw"), "B3_llrd": dict(llrd=llrd)}
    if base != "E6_qwen3_lora":
        abl["B1_lora"] = dict(method="lora", lr=3e-4, head_lr=1e-3)
    ABLATIONS[base] = {f"{prefix}{n}": {**EXPERIMENTS[base], **kw} for n, kw in abl.items()}
    EXPERIMENTS.update(ABLATIONS[base])

# Asymmetric Loss sobre SPECTER2: solo se usa en la combinación de modelos
EXPERIMENTS["A2_asl"] = {**EXPERIMENTS["E3_specter2"], "loss": "asl"}

# 4. SVM de referencia
La SVM final del capítulo anterior, recalculada partición a partición para medirla igual que los Transformers.

In [11]:
SVM_REF = dict(tipo="SVM", C=0.05, word_ngrams=(1, 2), char_ngrams=(2, 5), dedup="smart", max_features=300_000)

def svm_reference_run(name="SVM_ref", cfg=SVM_REF):
    save_config(name, cfg)
    texts = data["text"].to_numpy()
    keep = copies_to_keep(cfg["dedup"])
    for k, (tr, va) in enumerate(FOLDS):
        if fold_done(name, k):
            continue
        t0 = time.time()
        tr_rows = dev_idx[tr]
        vec = FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=cfg["word_ngrams"], min_df=2, max_df=0.9, max_features=cfg["max_features"],
                                     sublinear_tf=True, strip_accents="unicode")),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=cfg["char_ngrams"], min_df=3, max_df=0.9,
                                     max_features=cfg["max_features"], sublinear_tf=True, strip_accents="unicode"))])
        X_tr = vec.fit_transform(texts[tr_rows]).tocsr()
        rep = np.repeat(np.arange(len(tr_rows)), keep[tr_rows])
        X_rep = X_tr[rep]; X_rep.sort_indices()
        clf = OneVsRestClassifier(LinearSVC(C=cfg["C"], class_weight="balanced"), n_jobs=-1).fit(X_rep, Y[tr_rows][rep])
        def scores(txts):
            X = vec.transform(txts).tocsr(); X.sort_indices()
            return tie_alias(clf.decision_function(X))
        save_fold(name, k, scores(texts[dev_idx[va]]), scores(texts[ho_idx]), scores(TEST_TEXTS), time.time() - t0)

svm_reference_run()
display(results_table(["SVM_ref"]).round(4))

,experimento,modelo,folds,F1 fold0,±,F1 CV,± CV,precision,recall,accuracy,hamming,F1 holdout,horas
0,SVM_ref,SVM,3,0.3145,0.0087,0.3288,0.0049,0.2832,0.4406,0.4041,0.033,0.3277,0.3003


# 5. Representaciones congeladas
Cada texto se codifica una vez con el encoder sin ajustar. Sobre los vectores se entrena una regresión logística por clase (`class_weight="balanced"`) y un kNN con los 25 vecinos más similares.

In [12]:
EMB_MODELS = {
    "specter2":  dict(model="allenai/specter2_base", pooling="cls"),
    "bge-large": dict(model="BAAI/bge-large-en-v1.5", pooling="cls"),
    "e5-large":  dict(model="intfloat/e5-large-v2", pooling="mean", prefix="query: "),
    "qwen3-emb": dict(model="Qwen/Qwen3-Embedding-0.6B", pooling="last", padding_side="left", bs=32),
}
EMB_C_GRID = [0.1, 1.0]
KNN_K, KNN_TEMP = 25, 0.05

@torch.no_grad()
def encode_texts(ecfg, max_len=384):
    side = ecfg.get("padding_side", "right")
    tok = AutoTokenizer.from_pretrained(ecfg["model"])
    model = AutoModel.from_pretrained(ecfg["model"]).to(DEVICE).eval()
    if USE_AMP:
        model.half()
    texts = data["text"].tolist() + TEST_TEXTS
    seqs = [np.array(x, dtype=np.int32) for x in
            tok([ecfg.get("prefix", "") + t for t in texts], truncation=True, max_length=max_len)["input_ids"]]
    lens = np.array([len(s) for s in seqs])
    Z = np.zeros((len(seqs), model.config.hidden_size), np.float32)
    for b in make_batches(lens, ecfg.get("bs", 64), shuffle=False):
        ids, mask = collate([seqs[i] for i in b], pad_id_of(tok), side)
        h = model(input_ids=ids, attention_mask=mask).last_hidden_state
        if ecfg["pooling"] == "cls":
            z = h[:, 0]
        elif ecfg["pooling"] == "mean":
            m = mask.unsqueeze(-1).to(h.dtype)
            z = (h * m).sum(1) / m.sum(1)
        else:
            z = h[:, -1]
        Z[b] = F.normalize(z.float(), dim=-1).cpu().numpy()
    del model; gc.collect()
    return Z

def get_embeddings(name):
    path = os.path.join(EMB_DIR, f"{name}.npy")
    if os.path.exists(path):
        return np.load(path).astype(np.float32)
    Z = encode_texts(EMB_MODELS[name])
    np.save(path, Z.astype(np.float16))
    return Z

def fit_linear(Z_tr, Y_tr, Z_evals, C):
    sc = StandardScaler().fit(Z_tr)
    clf = OneVsRestClassifier(LogisticRegression(C=C, class_weight="balanced", max_iter=1000), n_jobs=-1)
    clf.fit(sc.transform(Z_tr), Y_tr)
    return [tie_alias(clf.decision_function(sc.transform(Z))) for Z in Z_evals]

def knn_scores(Z_tr, Y_tr, Z_q, k=KNN_K, temp=KNN_TEMP):
    out = np.zeros((len(Z_q), Y_tr.shape[1]), np.float32)
    for i in range(0, len(Z_q), 2048):
        sims = Z_q[i:i + 2048] @ Z_tr.T
        top = np.argpartition(-sims, k, axis=1)[:, :k]
        s = np.take_along_axis(sims, top, 1)
        w = np.exp((s - s.max(1, keepdims=True)) / temp)
        w /= w.sum(1, keepdims=True)
        out[i:i + 2048] = np.einsum("nk,nkc->nc", w, Y_tr[top])
    return tie_alias(out)

def embedding_runs(emb):
    names = [f"EMB_{emb}_C{C}" for C in EMB_C_GRID] + [f"KNN_{emb}"]
    if all(fold_done(n, k) for n in names for k in range(N_FOLDS)):
        return
    Z = get_embeddings(emb)
    Z_dev, Z_te = Z[:N], Z[N:]
    for k, (tr, va) in enumerate(FOLDS):
        Z_tr, Y_tr = Z_dev[dev_idx[tr]], Y[dev_idx[tr]]
        evals = [Z_dev[dev_idx[va]], Z_dev[ho_idx], Z_te]
        for C in EMB_C_GRID:
            name = f"EMB_{emb}_C{C}"
            save_config(name, {"tipo": "embeddings + LR", "model": EMB_MODELS[emb]["model"], "C": C})
            if not fold_done(name, k):
                t0 = time.time()
                save_fold(name, k, *fit_linear(Z_tr, Y_tr, evals, C), time.time() - t0)
        name = f"KNN_{emb}"
        save_config(name, {"tipo": "kNN", "model": EMB_MODELS[emb]["model"], "k": KNN_K, "temp": KNN_TEMP})
        if not fold_done(name, k):
            t0 = time.time()
            save_fold(name, k, *[knn_scores(Z_tr, Y_tr, Zq) for Zq in evals], time.time() - t0)

for emb in EMB_MODELS:
    embedding_runs(emb)

In [13]:
f1_cv = results_table(["SVM_ref"] + [f"EMB_{e}_C{C}" for e in EMB_MODELS for C in EMB_C_GRID]
                      + [f"KNN_{e}" for e in EMB_MODELS]).set_index("experimento")["F1 CV"]
tabla_41 = pd.DataFrame({
    "Regresión logística": {EMB_MODELS[e]["model"]: max(f1_cv[f"EMB_{e}_C{C}"] for C in EMB_C_GRID) for e in EMB_MODELS},
    "kNN": {EMB_MODELS[e]["model"]: f1_cv[f"KNN_{e}"] for e in EMB_MODELS}})
tabla_41.loc["SVM"] = [f1_cv["SVM_ref"], np.nan]
display(tabla_41.round(4))

,Regresión logística,kNN
allenai/specter2_base,0.3134,0.3132
BAAI/bge-large-en-v1.5,0.2848,0.3107
intfloat/e5-large-v2,0.2862,0.2980
Qwen/Qwen3-Embedding-0.6B,0.2992,0.3009
SVM,0.3288,NaN


# 6. Ajuste fino
Seis modelos con la misma receta, evaluados en la primera partición junto a la SVM.

In [14]:
for name in SCREEN:
    run_experiment(name, folds=[0])

tabla_42 = results_table(["SVM_ref"] + SCREEN)[["experimento", "modelo", "F1 fold0", "±"]]
display(tabla_42.sort_values("F1 fold0").round(4))

,experimento,modelo,F1 fold0,±
5,E5_deberta,microsoft/deberta-v3-base,0.2688,0.0017
0,SVM_ref,SVM,0.3145,0.0087
1,E1_roberta,roberta-base,0.3201,0.0003
4,E4_modernbert,answerdotai/ModernBERT-base,0.3238,0.0096
2,E2_scibert,allenai/scibert_scivocab_uncased,0.3394,0.0082
6,E6_qwen3_lora,Qwen/Qwen3-Embedding-0.6B,0.3428,0.0079
3,E3_specter2,allenai/specter2_base,0.3473,0.0128


# 7. Ablaciones
Cada modificación se aplica de una en una sobre los tres mejores modelos, en la primera partición. El cRT reentrena la cabeza de SPECTER2 como una regresión logística balanceada sobre sus vectores `[CLS]`.

In [15]:
for exps in ABLATIONS.values():
    for name in exps:
        run_experiment(name, folds=[0])

def crt_run(src, C=1.0):
    name = f"{src}_cRT"
    save_config(name, {"tipo": "cRT", "origen": src, "C": C})
    for k, (tr, va) in enumerate(FOLDS):
        fpath = os.path.join(run_dir(src), f"feat_fold{k}.npz")
        if fold_done(name, k) or not os.path.exists(fpath):
            continue
        t0 = time.time()
        f = np.load(fpath)
        S = fit_linear(f["tr"].astype(np.float32), Y[dev_idx[tr]],
                       [f["va"].astype(np.float32), f["ho"].astype(np.float32), f["te"].astype(np.float32)], C)
        save_fold(name, k, *S, time.time() - t0)

crt_run("E3_specter2")

In [16]:
abl_names = [n for exps in ABLATIONS.values() for n in exps]
f1_0 = results_table(list(TOP_MODELS) + abl_names + ["E3_specter2_cRT"]).set_index("experimento")["F1 fold0"]
VARIANTS = {"+ pos_weight": "A1_posweight", "+ LoRA": "B1_lora", "+ LLRD": "B3_llrd"}
tabla_43 = pd.DataFrame({
    TOP_NAMES[base]: {"Referencia": f1_0[base],
                                                **{v: f1_0.get(prefix + n, np.nan) for v, n in VARIANTS.items()},
                                                "+ cRT": f1_0.get(f"{base}_cRT", np.nan)}
    for base, (prefix, _) in TOP_MODELS.items()})
display(tabla_43.round(4))

,SPECTER2,Qwen3 + LoRA,SciBERT
Referencia,0.3473,0.3428,0.3394
+ pos_weight,0.3476,0.3652,0.3433
+ LoRA,0.3634,NaN,0.3583
+ LLRD,0.3515,0.3550,0.3397
+ cRT,0.2978,NaN,NaN


# 8. Modelos finales
Las mejores variantes se entrenan con las 3 particiones. La predicción del holdout y del test es la media de los tres modelos.

In [17]:
FINAL = ["E3_specter2", "A1_posweight", "B1_lora", "B3_llrd", "SB_B1_lora", "QW_A1_posweight", "A2_asl"]
for name in FINAL:
    run_experiment(name, folds=[0, 1, 2])

cols = ["experimento", "modelo", "F1 CV", "± CV", "precision", "recall", "accuracy", "hamming", "F1 holdout"]
tabla_44 = results_table(["SVM_ref"] + FINAL)[cols]
display(tabla_44.round(4))

,experimento,modelo,F1 CV,± CV,precision,recall,accuracy,hamming,F1 holdout
0,SVM_ref,SVM,0.3288,0.0049,0.2832,0.4406,0.4041,0.0330,0.3277
1,E3_specter2,allenai/specter2_base,0.3481,0.0055,0.2999,0.4758,0.4373,0.0312,0.3497
2,A1_posweight,allenai/specter2_base,0.3505,0.0043,0.3085,0.4297,0.4319,0.0307,0.3651
3,B1_lora,allenai/specter2_base,0.3444,0.0080,0.2911,0.4752,0.4371,0.0309,0.3686
4,B3_llrd,allenai/specter2_base,0.3519,0.0082,0.3054,0.4793,0.4395,0.0304,0.3665
5,SB_B1_lora,allenai/scibert_scivocab_uncased,0.3488,0.0068,0.3112,0.4519,0.4343,0.0310,0.3595
6,QW_A1_posweight,Qwen/Qwen3-Embedding-0.6B,0.3643,0.0018,0.3157,0.4517,0.4300,0.0308,0.3879
7,A2_asl,allenai/specter2_base,0.3465,0.0087,0.3002,0.4373,0.4402,0.0308,0.3624


In [18]:
# F1 por clase: SVM frente a SPECTER2 (3 particiones)
pc = pd.DataFrame({"SVM": evaluate_run(load_run("SVM_ref"))["per_class_f1"],
                   "SPECTER2": evaluate_run(load_run("E3_specter2"))["per_class_f1"],
                   "artículos dev": Y_dev.sum(0)}, index=CLASSES)
pc["diferencia"] = pc["SPECTER2"] - pc["SVM"]
display(pc.sort_values("artículos dev").round(3).T)

,physics.chem-ph,cs.HC,math.IT,cs.IT,stat.TH,math.ST,stat.AP,stat.ME,cs.SY,eess.SY,...,cs.SI,cs.CR,cs.NE,cs.CL,cs.RO,eess.IV,cs.AI,stat.ML,cs.LG,cs.CV
SVM,0.296,0.039,0.096,0.096,0.172,0.172,0.068,0.084,0.245,0.245,...,0.324,0.437,0.159,0.468,0.434,0.232,0.444,0.728,0.866,0.946
SPECTER2,0.192,0.058,0.142,0.142,0.153,0.153,0.112,0.131,0.264,0.264,...,0.360,0.532,0.202,0.509,0.483,0.247,0.457,0.733,0.866,0.951
artículos dev,17.000,34.000,36.000,36.000,37.000,37.000,39.000,40.000,59.000,59.000,...,224.000,233.000,433.000,598.000,897.000,1072.000,3823.000,8132.000,14591.000,16819.000
diferencia,-0.104,0.018,0.046,0.046,-0.019,-0.019,0.044,0.047,0.019,0.019,...,0.036,0.095,0.042,0.041,0.049,0.016,0.013,0.005,0.001,0.005


# 9. Envíos a Codabench
SPECTER2 con LLRD y Qwen3 con LoRA y `pos_weight`, con los umbrales ajustados sobre todas las predicciones *out-of-fold*.

In [19]:
SUBMITS = ["B3_llrd", "QW_A1_posweight"]

def make_submission(name):
    R = load_run(name)
    thr = tune_thresholds(Y_dev, R["oof"])
    P_te = decode_flat(R["te"], thr)
    sub = test[["titles", "summaries"]].copy()
    sub["terms"] = [str(list(t)) for t in mlb.inverse_transform(P_te)]
    sub_dir = os.path.join(OUT_DIR, f"sub_LC-{name}")
    os.makedirs(sub_dir, exist_ok=True)
    csv_path = os.path.join(sub_dir, "dataset_test_preds.csv")
    sub.to_csv(csv_path, index=False)
    with zipfile.ZipFile(os.path.join(sub_dir, f"LC-{name}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname="dataset_test_preds.csv")
    np.save(os.path.join(sub_dir, "test_scores.npy"), R["te"])
    np.save(os.path.join(sub_dir, "thresholds.npy"), thr)
    assert len(sub) == len(test) and sub["terms"].map(lambda s: len(ast.literal_eval(s)) > 0).all()
    print(f"{name}: {P_te.sum(1).mean():.2f} etiquetas por artículo en test")

for name in SUBMITS:
    make_submission(name)

B3_llrd: 2.21 etiquetas por artículo en test
QW_A1_posweight: 2.22 etiquetas por artículo en test
